# Phase 2 — Feature Engineering & Preprocessing
## FCM Gradational Lithofacies Classification
**Author:** Kumar Yuvraj (23GG5PE02) | IIT Kharagpur

FCM is a distance-based algorithm: if GR ranges 0–200 API and RHOB ranges 1.8–3.0 g/cc,  
GR differences dominate Euclidean distance and RHOB is effectively ignored.  
**Every preprocessing decision here directly shapes the cluster geometry.**

Steps:
1. NaN handling — drop rows with >2 missing logs; interpolate per-well along depth  
2. Log-transform RDEP only (resistivity is log-normally distributed)  
3. StandardScaler on the 5-feature matrix  
4. Pairplot on a representative well (pre-FCM cluster separability check)

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler
os.chdir('D:/Projects/fcm-lithofacies')

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
MASTER_CSV = './outputs/data/force2020_master.csv'
OUT_DATA   = './outputs/data/'
OUT_FIGS   = './outputs/figures/'
os.makedirs(OUT_DATA, exist_ok=True)
os.makedirs(OUT_FIGS, exist_ok=True)

LOG_COLS   = ['GR', 'RHOB', 'NPHI', 'DTC', 'RDEP']   # 5 primary logs
FEAT_COLS  = ['GR', 'RHOB', 'NPHI', 'DTC', 'RDEP_LOG']  # after transform

master = pd.read_csv(MASTER_CSV, low_memory=False)
print(f'Loaded: {len(master):,} rows, {master["WELL"].nunique()} wells')
print(f'Columns: {list(master.columns)}')
print(f'NaN counts (before any cleaning):')
print(master[LOG_COLS].isna().sum().to_string())

Loaded: 1,170,511 rows, 98 wells
Columns: ['WELL', 'DEPTH_MD', 'GR', 'RHOB', 'NPHI', 'DTC', 'RDEP', 'LITH_LABEL']
NaN counts (before any cleaning):
GR           0
RHOB    161269
NPHI    405102
DTC      80863
RDEP     11015


In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — NaN Handling
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY drop rows with >2 missing logs:
# FCM uses all 5 features per sample.  If 3+ logs are missing simultaneously,
# interpolation would fabricate >60% of that row's information — the result
# is a synthetic data point, not real geology.

n_before = len(master)
nan_count_per_row = master[LOG_COLS].isna().sum(axis=1)
master = master[nan_count_per_row <= 2].copy()
n_after_drop = len(master)
print(f'Step 1a — Drop rows with >2 missing logs:')
print(f'  Before: {n_before:,}  →  After: {n_after_drop:,}  '
      f'(dropped {n_before - n_after_drop:,} rows)')

# ── Per-well depth interpolation ──────────────────────────────────────────
# WHY per-well (not global):
# A global interpolate() or fillna() mixes geology across different wells.
# GR=80 in well A means shale; GR=80 in well B (lower baseline) might mean
# sandstone.  Interpolating within each well respects the local baseline.

# Sort within each well by depth before interpolation
master = master.sort_values(['WELL', 'DEPTH_MD']).reset_index(drop=True)

for log in LOG_COLS:
    # groupby('WELL') → interpolate within each well along depth index
    master[log] = (
        master
        .groupby('WELL')[log]
        .transform(lambda s: s.interpolate(method='linear', limit_direction='both'))
    )
    # Any residual NaN (whole-well missing) → per-well median fallback
    master[log] = (
        master
        .groupby('WELL')[log]
        .transform(lambda s: s.fillna(s.median()))
    )
    # Final fallback: global median (rare — only if a whole log is absent in a well)
    master[log] = master[log].fillna(master[log].median())

n_nan_after = master[LOG_COLS].isna().sum().sum()
print(f'\nStep 1b — Per-well depth interpolation (groupby WELL):')
print(f'  Remaining NaN across all 5 logs: {n_nan_after}')
assert n_nan_after == 0, 'Unexpected NaN after imputation!'
print('  ✓ Zero NaN confirmed')
import os
os.chdir('D:/Projects/fcm-lithofacies')   # set project root

# Ensure all output directories exist
os.makedirs('outputs/data',    exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)

print("Working dir:", os.getcwd())

Step 1a — Drop rows with >2 missing logs:
  Before: 1,170,511  →  After: 1,127,735  (dropped 42,776 rows)

Step 1b — Per-well depth interpolation (groupby WELL):
  Remaining NaN across all 5 logs: 0
  ✓ Zero NaN confirmed
Working dir: D:\Projects\fcm-lithofacies


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Log Transformation (RDEP only)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY only RDEP is log-transformed:
#   Resistivity spans several orders of magnitude (0.2 → 2000 Ω·m in FORCE 2020).
#   Its distribution is strongly right-skewed and empirically log-normal
#   in clastic sequences.  In raw form, a single ultra-high-resistivity
#   carbonate interval can dominate the distance metric for every sample
#   in the dataset.  log10(RDEP) compresses this to a ~3-unit range.
#
#   GR, RHOB, NPHI, DTC are NOT log-transformed because:
#   - GR: approximately normal within-formation; units are linear (API)
#   - RHOB: tight physical range (1.8–3.0 g/cc), already near-normal
#   - NPHI: bounded [–0.15, 1.0], no long tail
#   - DTC: somewhat skewed but transformation would obscure the
#           physically meaningful acoustic impedance relationships

master['RDEP_LOG'] = np.log10(master['RDEP'].clip(0.01, None))

print('RDEP before transform (sample statistics):')
print(master['RDEP'].describe())
print()
print('RDEP_LOG after log10 transform:')
print(master['RDEP_LOG'].describe())

# Verify range is compressed
r_range = master['RDEP'].max() - master['RDEP'].min()
l_range = master['RDEP_LOG'].max() - master['RDEP_LOG'].min()
print(f'\nRange compression: RDEP span={r_range:.1f} Ω·m → '
      f'RDEP_LOG span={l_range:.2f} log-units')

RDEP before transform (sample statistics):
count    1.127735e+06
mean     1.102286e+01
std      1.159297e+02
min      3.170056e-02
25%      9.283628e-01
50%      1.459855e+00
75%      2.596859e+00
max      1.999887e+03
Name: RDEP, dtype: float64

RDEP_LOG after log10 transform:
count    1.127735e+06
mean     2.491091e-01
std      4.398044e-01
min     -1.498933e+00
25%     -3.228229e-02
50%      1.643097e-01
75%      4.144483e-01
max      3.301005e+00
Name: RDEP_LOG, dtype: float64

Range compression: RDEP span=1999.9 Ω·m → RDEP_LOG span=4.80 log-units


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — StandardScaler
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY StandardScaler (z-score) not MinMaxScaler:
# MinMaxScaler is sensitive to outliers — a single erroneous GR=300 would
# compress all other GR values into a tiny range.  StandardScaler is more
# robust because it is only sensitive to mean and std, not to extreme values.
# Per-feature z-scoring also preserves the relative spread within each log,
# which is geologically meaningful (high GR variance = more heterogeneous).

X = master[FEAT_COLS].values   # shape (N, 5)

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler for reuse on new wells later
scaler_path = os.path.join(OUT_DATA, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f'✓ Scaler saved: {scaler_path}')

# Save scaled matrix
X_path = os.path.join(OUT_DATA, 'X_scaled.npy')
np.save(X_path, X_scaled)
print(f'✓ X_scaled saved: {X_path}  shape={X_scaled.shape}')

# Verify scaling
print(f'\nPost-scaling means (should be ≈ 0):')
print({f: f'{v:.4f}' for f, v in zip(FEAT_COLS, X_scaled.mean(axis=0))})
print(f'Post-scaling stds (should be ≈ 1):')
print({f: f'{v:.4f}' for f, v in zip(FEAT_COLS, X_scaled.std(axis=0))})

# Also save the full DataFrame with LITH_LABEL for downstream labelling
master.to_csv(os.path.join(OUT_DATA, 'force2020_preprocessed.csv'), index=False)
print('\n✓ force2020_preprocessed.csv saved (with RDEP_LOG column)')

✓ Scaler saved: ./outputs/data/scaler.pkl
✓ X_scaled saved: ./outputs/data/X_scaled.npy  shape=(1127735, 5)

Post-scaling means (should be ≈ 0):
{'GR': '0.0000', 'RHOB': '-0.0000', 'NPHI': '-0.0000', 'DTC': '0.0000', 'RDEP_LOG': '0.0000'}
Post-scaling stds (should be ≈ 1):
{'GR': '1.0000', 'RHOB': '1.0000', 'NPHI': '1.0000', 'DTC': '1.0000', 'RDEP_LOG': '1.0000'}

✓ force2020_preprocessed.csv saved (with RDEP_LOG column)


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Exploratory Pairplot (most complete well)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY pairplot before FCM:
# The pairplot shows whether the 5 logs can visually separate lithofacies.
# If classes cluster cleanly → FCM will likely succeed.
# Heavy overlap (especially in the GR–RHOB space) justifies fuzzy assignment
# rather than hard K-Means: the samples physically lie in transition zones.

# Find the well with most complete log coverage
completeness = (
    master.groupby('WELL')[LOG_COLS]
    .apply(lambda df: df.notna().all(axis=1).sum())
)
best_well = completeness.idxmax()
print(f'Most complete well: {best_well}  ({int(completeness[best_well]):,} complete rows)')

wdf = master[master['WELL'] == best_well].copy()

# Build scaled feature DataFrame for this well
X_well = scaler.transform(wdf[FEAT_COLS].values)
plot_df = pd.DataFrame(X_well, columns=[f+'_z' for f in FEAT_COLS])
if 'LITH_LABEL' in wdf.columns:
    plot_df['Lithology'] = wdf['LITH_LABEL'].values
else:
    plot_df['Lithology'] = 'Unknown'

# Sub-sample for speed (max 5000 points)
if len(plot_df) > 5000:
    plot_df = plot_df.sample(5000, random_state=42)

# ── Build palette from the most common lithologies ─────────────────────────
LITHO_PALETTE = {
    'Sandstone':       '#DDBB00',
    'Sandstone/Shale': '#FF8C00',
    'Shale':           '#707070',
    'Marl':            '#228B22',
    'Dolomite':        '#1E90FF',
    'Limestone':       '#87CEEB',
    'Coal':            '#1a1a1a',
    'Chalk':           '#C8A000',
    'Halite':          '#FF69B4',
    'Tuff':            '#A0522D',
    'Anhydrite':       '#9370DB',
    'Basement':        '#8B0000',
}
present_palette = {k: v for k, v in LITHO_PALETTE.items()
                   if k in plot_df['Lithology'].unique()}

g = sns.pairplot(
    plot_df,
    hue='Lithology',
    palette=present_palette,
    plot_kws={'alpha': 0.3, 's': 8},
    diag_kind='kde',
)
g.figure.suptitle(
    f'Scaled Log Pairplot — Well: {best_well}\n'
    f'(Pre-FCM: heavy overlap = gradational transitions justify fuzzy clustering)',
    y=1.02, fontsize=11, fontweight='bold'
)

fig_path = os.path.join(OUT_FIGS, 'pairplot_representative_well.png')
g.figure.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'✓ Saved: {fig_path}')

Most complete well: 25/2-7  (25,131 complete rows)
✓ Saved: ./outputs/figures/pairplot_representative_well.png


## Pairplot Interpretation

The pairplot above shows the 5 scaled wireline logs coloured by FORCE 2020 lithofacies labels for the most data-complete well.

**Key observations:**
- **GR vs RHOB** (top-left region): The sandstone cloud (yellow) sits at low GR / moderate-high RHOB, while shale (grey) occupies high GR / lower RHOB.  However, a large Sandstone/Shale population (orange) bridges the two — these are the gradational transition samples that no GR hard cutoff can cleanly separate.
- **NPHI vs RHOB**: The neutron-density crossplot shows a classic 'fan' shape.  Gas sands deviate toward high NPHI / low RHOB (the crossover zone), while brine sands and shales separate more clearly along the RHOB axis.  The overlap again confirms that fuzzy assignment is more physically appropriate than K-Means.
- **DTC vs GR**: Compressional slowness increases with clay content and decreasing cementation.  The shale population spans a wide DTC range, reflecting variable compaction — a hard classifier would split this into multiple 'shale' classes unnecessarily.
- **RDEP_LOG**: Resistivity separates coal (highest) and carbonates (high) from shale (lowest) most cleanly.  But within sandstone, resistivity varies by fluid type — this is the one log where class overlap is almost unavoidable without fluid-type information.

**Conclusion:** The substantial inter-class overlap in every log pair confirms that the geological reality is gradational.  FCM will assign partial memberships to these transition-zone samples rather than forcing them into a single discrete class — which is physically correct for fining-upward fluvial sequences like the Barakar Formation.